# PDF Data Extraction Pipeline
## LandingAI ADE Document Parsing

This notebook handles the **extraction phase** of the data pipeline. It:
- Discovers PDFs across all cities in `input_folder/`
- Uses LandingAI ADE SDK for table extraction
- Saves raw JSON and CSV files in city-specific folders
- Is **generic** and works for any city without modification

**Purpose**: Extract structured data from violation PDFs, leaving city-specific cleaning for separate notebooks.

## Environment Setup & Project Paths
Set the project root and make the src package importable from this notebook.

In [1]:
# --- Notebook bootstrap (root, sys.path, masked prints) ---
import sys, os
from pathlib import Path

# Use your existing mask_path() if present; otherwise a tiny fallback
if "mask_path" not in globals():
    def mask_path(p: str | Path) -> str:
        p = Path(p)
        parts = p.parts
        # show only the last 3 segments, prefix with "..."
        return str(Path(*(["..."] + list(parts[-3:]))))

# Robust ROOT detection (works whether notebook lives in src/ or project root)
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    cwd = Path.cwd()
    ROOT = cwd.parent if cwd.name == "src" else cwd

SRC_DIR = ROOT / "src"

# Ensure both ROOT and src/ are importable
for path in (ROOT, SRC_DIR):
    s = str(path)
    if s not in sys.path:
        sys.path.insert(0, s)

# Project imports
import src.utils as u
import src.ade_client as ac

print("ROOT:", mask_path(ROOT))
print("SRC_DIR:", mask_path(SRC_DIR))
print("utils.to_jsonable:", hasattr(u, "to_jsonable"))
print("ade_client.parse_pdf:", hasattr(ac, "parse_pdf"))

ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
SRC_DIR: ...\LandingAI-Hack\coderisk-sf\src
utils.to_jsonable: True
ade_client.parse_pdf: True


## Load Environment Variables and Initialize ADE Client
Load environment variables from the .env file and initialize the LandingAI ADE client using the API key.

In [2]:
import os
from dotenv import load_dotenv
from landingai_ade import LandingAIADE

# Load .env file from project root
env_path = ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f".env not found at {env_path}")
load_dotenv(env_path)

# Retrieve API key
api_key = os.getenv("VISION_AGENT_API_KEY")

if not api_key:
    raise ValueError(
        "LandingAI API key not found. "
        "Add VISION_AGENT_API_KEY to your .env file."
    )

# Initialize ADE client
ade_client = LandingAIADE(apikey=api_key)

print("ADE client initialized successfully!")

ADE client initialized successfully!


## Discover PDFs Across All Cities
Scan `input_folder/` for city subfolders and discover all PDFs. This is completely generic and works for any city structure.

In [ ]:
from pathlib import Path
from collections import Counter

# Base directories
INPUT_DIR = ROOT / "input_folder"
RESULTS_DIR = ROOT / "results_folder"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def normalize_city_name(folder_name: str) -> str:
    """Standardize folder names into clean city labels."""
    return (
        folder_name.replace("_", " ")
                   .replace("-", " ")
                   .strip() 
    )

def discover_city_pdfs(input_root: Path):
    """
    Scan input_folder/<city>/ for PDFs.
    Returns a list of (city_name, pdf_path).
    """
    items = []
    if not input_root.exists():
        print(f"Input directory not found: {input_root}")
        return items

    for sub in sorted(p for p in input_root.iterdir() if p.is_dir()):
        city = normalize_city_name(sub.name)
        pdfs = sorted(sub.glob("*.pdf"))

        # Create matching results folder automatically (using lowercase folder name)
        folder_name = sub.name.lower()  # Force lowercase for consistency
        city_results = RESULTS_DIR / folder_name / "raw_json"
        city_results.mkdir(parents=True, exist_ok=True)

        for pdf in pdfs:
            items.append((city, pdf))

    return items

# Discover PDFs
city_pdfs = discover_city_pdfs(INPUT_DIR)

# Inventory summary
counts = Counter([city for city, _ in city_pdfs])
print(f"Found {len(city_pdfs)} PDFs across {len(counts)} cities in {mask_path(INPUT_DIR)}")

for city, n in counts.items():
    print(f" - {city:<20} {n} PDFs")

# Preview first few items
if city_pdfs:
    print("\nPreview:")
    for city, p in city_pdfs[:5]:
        print(f"   {city:<20} -> {p.name}")
else:
    print("No PDFs found.")

Found 5 PDFs across 5 cities in ...\LandingAI-Hack\coderisk-sf\input_folder
 - Bocaraton            1 PDFs
 - Oaklandpark          1 PDFs
 - Pompano              1 PDFs
 - Tamarac              1 PDFs
 - Wiltonmanor          1 PDFs

Preview:
   Bocaraton            -> boca_JustFOIA_Request_2024-9180-30p.pdf
   Oaklandpark          -> oakland_public_request_2024-041_Code_Cases_Violation-30p.pdf
   Pompano              -> pompanoViolations_20240124-30p.pdf
   Tamarac              -> TAMARAC_PRR-265-2024.pdf
   Wiltonmanor          -> wilton_Code_Violation_to_March_2024-30p.pdf


## Generic Extraction Pipeline
Extract tables from all discovered PDFs using ADE SDK. Saves raw JSON and CSV files in city-specific folders.
This is **resumable** - already processed files are skipped.

In [4]:
import json
import pandas as pd
from src.ade_client import parse_pdf, extract_cases_df

def ensure_city_dirs(city: str, pdf_path: Path) -> tuple[Path, Path]:
    """Return (raw_json_dir, tables_dir) for a city; create if missing."""
    # Use lowercase folder name for consistency
    folder_name = pdf_path.parent.name.lower()  # Force lowercase
    city_root   = RESULTS_DIR / folder_name
    raw_json    = city_root / "raw_json"
    tables_dir  = city_root / "tables"
    raw_json.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    return raw_json, tables_dir

def save_json(payload: dict, path: Path) -> None:
    """Save JSON with proper encoding."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# Resume support - check what's already been processed
processed_files = set()
for city_folder in RESULTS_DIR.iterdir():
    if city_folder.is_dir():
        json_dir = city_folder / "raw_json"
        if json_dir.exists():
            for json_file in json_dir.glob("*.json"):
                processed_files.add(json_file.stem)

print(f"Found {len(processed_files)} already processed files. Skipping...")

# Main extraction loop
success_count = 0
error_count = 0

for city, pdf_path in city_pdfs:
    # Skip if already processed
    if pdf_path.stem in processed_files:
        print(f"[skip] {city:<12} :: {pdf_path.name} (already processed)")
        continue

    try:
        # 1) Parse with ADE
        print(f"[processing] {city:<12} :: {pdf_path.name}")
        parsed = parse_pdf(pdf_path)

        # 2) Save raw JSON for auditability
        raw_json_dir, tables_dir = ensure_city_dirs(city, pdf_path)
        save_json(parsed, raw_json_dir / f"{pdf_path.stem}.json")

        # 3) Convert tables -> DataFrame
        df_raw = extract_cases_df(parsed)
        if df_raw.empty:
            print(f"[warning] No table rows extracted: {city} :: {pdf_path.name}")
            continue

        # 4) Attach metadata and write per-file CSV
        df_raw["city"] = city
        df_raw["source_file"] = pdf_path.name
        df_raw.to_csv(tables_dir / f"{pdf_path.stem}.csv", index=False)

        print(f"[success] {city:<12} :: {pdf_path.name} (+{len(df_raw)} rows)")
        success_count += 1

    except Exception as e:
        print(f"[error] {city:<12} :: {pdf_path.name} -> {e}")
        error_count += 1

print(f"\n✅ Extraction complete: {success_count} successful, {error_count} errors")
print(f"📁 Raw data saved in: {mask_path(RESULTS_DIR)}")
print(f" Next step: Run city-specific cleaning notebooks (2_cleaning_*.ipynb)")

Found 1 already processed files. Skipping...
[skip] Bocaraton    :: boca_JustFOIA_Request_2024-9180-30p.pdf (already processed)
[processing] Oaklandpark  :: oakland_public_request_2024-041_Code_Cases_Violation-30p.pdf
[success] Oaklandpark  :: oakland_public_request_2024-041_Code_Cases_Violation-30p.pdf (+1224 rows)
[processing] Pompano      :: pompanoViolations_20240124-30p.pdf
[success] Pompano      :: pompanoViolations_20240124-30p.pdf (+419 rows)
[processing] Tamarac      :: TAMARAC_PRR-265-2024.pdf
[success] Tamarac      :: TAMARAC_PRR-265-2024.pdf (+66 rows)
[processing] Wiltonmanor  :: wilton_Code_Violation_to_March_2024-30p.pdf
[success] Wiltonmanor  :: wilton_Code_Violation_to_March_2024-30p.pdf (+568 rows)

✅ Extraction complete: 4 successful, 0 errors
📁 Raw data saved in: ...\LandingAI-Hack\coderisk-sf\results_folder
 Next step: Run city-specific cleaning notebooks (2_cleaning_*.ipynb)
